# GPU 利用率：为什么一直 0%

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-05-gpu-utilization.ipynb)

动手理解 GPU-Util。用一段 GPU 密集计算让利用率从 0% 冲到 100%，搞清楚它到底采样什么、为什么 CPU↔GPU 拷贝时它可能还是 0%，纠正“利用率=在干活”的误解。

本 notebook 配合教程章节 [GPU 利用率：为什么一直 0%](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-05-gpu-utilization/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

`nvidia-smi` 的 `GPU-Util` 是 [Lab 01](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-01-meet-your-gpu/) 留下的另一个悬念。它常被当成“GPU 有没有在干活”的指标，但它其实只在采样窗口内反映“**有没有 kernel 在 SM 上执行**”。这会带来两类误判：明明在等数据搬运却显示 0%、或者短促 kernel 之间因采样没踩中而低估。搞清楚它采的到底是什么，才能在 [Lab 06 NVML](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-06-nvml/) 和后面的 HAMi 监控里正确读指标。

## 第 1 步：基线，空闲就是 0%

In [ ]:
import torch, subprocess, time

def util():
    out = subprocess.check_output(["nvidia-smi","--query-gpu=utilization.gpu","--format=csv,noheader,nounits"]).decode().strip()
    return out.splitlines()[0]

print("idle util:", util(), "%")

预期：`0%`。

## 第 2 步：让 GPU 真正忙起来

In [ ]:
size = 8192
a = torch.randn(size, size, device="cuda")
b = torch.randn(size, size, device="cuda")

# 在一个独立线程里持续 matmul，同时在主线程采样
from threading import Thread
running = {"on": True}
def busy():
    while running["on"]:
        c = a @ b
        torch.cuda.synchronize()

t = Thread(target=busy); t.start()
for _ in range(5):
    time.sleep(0.3)
    print("busy util:", util(), "%")
running["on"] = False
t.join()

预期：`busy util` 多次接近 **90–100%**。这就是“SM 上持续有 kernel 执行”的样子。顺便算一笔账：8192³ × 2 ≈ 1.1 TFLOP，T4 的 FP32 峰值约 8.1 TFLOPS，所以一次 matmul 约 130–150 ms，足够让 1 秒采样窗口里的每一刻都有 kernel 在跑。

## 第 3 步：制造“在搬数据但利用率为 0”

In [ ]:
import numpy as np
running2 = {"on": True}
def copy_only():
    while running2["on"]:
        # 只在 CPU<->GPU 之间搬运，不在 GPU 上算
        x = torch.randn(4096, 4096)
        gx = x.to("cuda")
        _ = gx.to("cpu")

t2 = Thread(target=copy_only); t2.start()
for _ in range(5):
    time.sleep(0.3)
    print("copy util:", util(), "%")
running2["on"] = False
t2.join()

预期：`copy util` 往往 **明显低于 busy**，尽管数据在密集搬运（PCIe 忙、显存在读写）。它不一定是 0：每次 `x.to("cuda")` 背后的 `cudaMemcpy` 不算 kernel，但这段代码的瓶颈其实在 CPU 端的 `torch.randn`，GPU 大部分时间在等 CPU。

> **关键洞察**
>
> `GPU-Util` 只看 **SM（流式多处理器）是否在执行 kernel**。纯数据搬运、纯 `cudaMemcpy`、CPU 端瓶颈、kernel 调度间隙，都可能让它是 0%。所以：
>
> - **利用率 0% ≠ 没干活**（可能在搬数据、可能在等 CPU）；
> - **利用率 100% ≠ 已经饱和**（SM 忙，但显存带宽、Tensor Core 可能还有余量）。
>
> 这也是为什么 HAMi 的 `nvidia.com/gpucores`（算力百分比限制）要和真正的“算力利用率”结合起来看，不能只盯一个数字。HAMi-core 正是通过周期性读取这个利用率，再在 CUDA API 层节流 kernel 发射来逼近目标百分比的。

## 第 4 步：采样窗口的影响

In [ ]:
# 跑一个间歇 kernel：约 1ms 计算 + 9ms 空闲，循环
# 1024³×2 ≈ 2 GFLOP，在 T4 上约 0.3–1 ms；不要用第 2 步的 8192，那要 100 ms 以上
sa = torch.randn(1024, 1024, device="cuda")
sb = torch.randn(1024, 1024, device="cuda")
running3 = {"on": True}
def burst():
    while running3["on"]:
        _ = sa @ sb
        torch.cuda.synchronize()
        time.sleep(0.009)

t3 = Thread(target=burst); t3.start()
for _ in range(8):
    time.sleep(0.4)
    print("burst util:", util(), "%")
running3["on"] = False
t3.join()

预期：波动较大，可能在 5–30% 之间。`GPU-Util` 的定义是“过去一个采样周期内，**有至少一个 kernel 在执行的时间占比**”，采样周期在 1/6 秒到 1 秒之间因产品而异。所以 10% duty cycle 的负载理论上读数应该在 10% 附近，但因为每次 kernel 的实际耗时、launch 开销和采样边界都不对齐，读数会来回跳。

## 验证

完成后你应该能：

- 说出 `GPU-Util` 实际采样什么（SM 是否在执行 kernel）；
- 解释为什么纯拷贝时利用率可能为 0；
- 说明为什么利用率 100% 不等于显存/Tensor Core 饱和；
- 知道采样窗口如何影响读数。

## 故障模式

| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| `busy util` 没到 100% | matmul 规模太小 / 采样踩到了 synchronize 间隙 | 加大 `size` 到 12288（注意显存：两个 12288² fp32 约 1.1 GiB） |
| `copy util` 不是 0 | 搬运夹带了小 kernel，或 `randn` 在 GPU 上执行 | 正常现象，重点是明显低于 busy |
| `busy util` 一直在 0 | runtime 是 CPU / 线程没真正跑 | 确认 T4 runtime；`torch.cuda.synchronize()` 必须在循环内 |
| `CUDA out of memory` | 第 2 步 的 a/b 和 Lab 04 残留叠加 | Runtime → Restart session 后重跑 |

## 跑完之后

如果你看到的输出与教程里的预期明显不符，或者碰到了教程没有列出的故障模式，欢迎带上驱动 / CUDA 版本和关键输出在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提 Issue 或 PR。